# BrainBridge - Teste de Quadrado Latino para Hiperparâmetros
## Busca de Variações Otimizadas: Canais vs Amostras (Window Size)

**Objetivo:** Testar 50 combinações aleatórias de:
- **Canais**: 4 a 16 canais EEG
- **Tamanho da Janela**: 1 a 4 segundos (125 a 500 amostras @ 125Hz)

**Saída esperada:**
- Scatter plot visualizando a região de possibilidades
- Identificação de ranges de loss otimizados
- Análise de qual combinação de parâmetros oferece melhor performance

In [1]:
import os, sys, glob, csv, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score, accuracy_score, precision_score, recall_score

# scipy para filtros
from scipy.signal import butter, filtfilt, sosfiltfilt

# tensorflow / keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, BatchNormalization, MaxPooling1D, Dropout, Flatten, Dense
from tensorflow.keras.optimizers import Adam

%matplotlib inline

print("✓ Todas as bibliotecas importadas com sucesso!")

✓ Todas as bibliotecas importadas com sucesso!


## 1️⃣ Configuração e Definição do Espaço de Busca

In [2]:
# ==== CONFIGURAÇÃO GLOBAL ====
FS = 125.0                          # Frequência de amostragem (Hz)
EPOCHS = 50                          # Epochs por experimento (EarlyStopping reduzirá)
BATCH_SIZE = 32
TEST_SIZE = 0.2
RANDOM_SEED = 42
EARLY_STOPPING_PATIENCE = 10

# Define o caminho para os dados
# ===== OPÇÃO 1: LAPTOP LOCAL (ATIVO) =====
DATA_DIR = r"C:\Users\Chari\Documents\dev\Brainbridge\tools\downloader\data\MNE-eegbci-data\files\eegmmidb\1.0.0"
PATTERN = "S*/*.csv"

# ===== OPÇÃO 2: GOOGLE COLAB (COMENTADO) =====
# DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/1.0.0"
# PATTERN = "S*/*.csv"

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Define o espaço de busca
num_channels_range = (4, 16)                    # 4 a 16 canais
window_size_samples_range = (int(1*FS), int(4*FS))  # 1 a 4 segundos (125 a 500 amostras)
num_experiments = 50                             # 50 variações aleatórias

print(f"📊 CONFIGURAÇÃO DO TESTE:")
print(f"   • Canais: {num_channels_range[0]} a {num_channels_range[1]}")
print(f"   • Tamanho de janela: {window_size_samples_range[0]} a {window_size_samples_range[1]} amostras ({window_size_samples_range[0]/FS:.1f}s a {window_size_samples_range[1]/FS:.1f}s)")
print(f"   • Frequência de amostragem: {FS} Hz")
print(f"   • Variações aleatórias: {num_experiments}")
print(f"   • Diretório de dados: {DATA_DIR}")

📊 CONFIGURAÇÃO DO TESTE:
   • Canais: 4 a 16
   • Tamanho de janela: 125 a 500 amostras (1.0s a 4.0s)
   • Frequência de amostragem: 125.0 Hz
   • Variações aleatórias: 50
   • Diretório de dados: C:\Users\Chari\Documents\dev\Brainbridge\tools\downloader\data\MNE-eegbci-data\files\eegmmidb\1.0.0


## 2️⃣ Definição de Funções Auxiliares

In [3]:
def create_windows_openbci(csv_files, fs=125.0, window_size=250, step=125):
    """Extrai APENAS a primeira janela de cada segmento T1..T0 ou T2..T0.
    
    Pega do T1/T2 até completar window_size amostras (primeira janela).
    Não faz sliding window, só a primeira de cada segmento.
    
    Retorna: X (n_samples, window_size, 16), y (n_samples,)
    """
    X_list, y_list = [], []
    
    for path in csv_files:
        data, markers = load_openbci_csv(path)
        if data.size == 0:
            continue
        idxs = find_marker_indices(markers)
        seg_T1 = segments_between(data, idxs.get("T1", []), idxs.get("T0", []))
        seg_T2 = segments_between(data, idxs.get("T2", []), idxs.get("T0", []))
        
        def add_first_window(seg_list, label):
            """Extrai APENAS a primeira janela de window_size amostras de cada segmento."""
            nonlocal X_list, y_list
            for seg in seg_list:
                n = len(seg)
                # Pega só se o segmento tem pelo menos window_size amostras
                if n >= window_size:
                    w = seg[:window_size]  # APENAS a primeira janela (T1 até T1+window_size)
                    X_list.append(w.astype(np.float32))
                    y_list.append(label)
        
        add_first_window(seg_T1, 0)
        add_first_window(seg_T2, 1)
    
    if not X_list:
        return np.zeros((0, window_size, 16), dtype=np.float32), np.zeros((0,), dtype=np.int32)
    X = np.stack(X_list).astype(np.float32)
    y = np.array(y_list, dtype=np.int32)
    return X, y

## 3️⃣ Carregamento de Dados e Geração de Combinações Aleatórias

In [4]:
# Busca arquivos CSV
csv_files = sorted(glob.glob(str(Path(DATA_DIR) / PATTERN)))
print(f"🔍 CSVs encontrados: {len(csv_files)}")
if csv_files:
    print("   Primeiros 5 arquivos:")
    for f in csv_files[:5]:
        print(f"   • {Path(f).name}")
else:
    print("⚠️ NENHUM CSV ENCONTRADO! Verifique DATA_DIR e PATTERN.")

# Gera 50 combinações aleatórias de (canais, window_size)
random_params = []
for _ in range(num_experiments):
    n_ch = np.random.randint(num_channels_range[0], num_channels_range[1] + 1)
    w_size = np.random.randint(window_size_samples_range[0], window_size_samples_range[1] + 1)
    random_params.append({'num_channels': n_ch, 'window_size': w_size})

print(f"\n🎲 {num_experiments} combinações aleatórias geradas:")
print("   Primeiras 10:")
for i, p in enumerate(random_params[:10]):
    print(f"   {i+1:2d}. Canais={p['num_channels']:2d}, Janela={p['window_size']:3d} amostras ({p['window_size']/FS:.2f}s)")

🔍 CSVs encontrados: 319
   Primeiros 5 arquivos:
   • S001R03_csv_openbci.csv
   • S001R04_csv_openbci.csv
   • S001R07_csv_openbci.csv
   • S001R08_csv_openbci.csv
   • S001R11_csv_openbci.csv

🎲 50 combinações aleatórias geradas:
   Primeiras 10:
    1. Canais=10, Janela=473 amostras (3.78s)
    2. Canais=14, Janela=196 amostras (1.57s)
    3. Canais=16, Janela=145 amostras (1.16s)
    4. Canais=10, Janela=246 amostras (1.97s)
    5. Canais= 6, Janela=339 amostras (2.71s)
    6. Canais=14, Janela=212 amostras (1.70s)
    7. Canais= 8, Janela=224 amostras (1.79s)
    8. Canais=11, Janela=276 amostras (2.21s)
    9. Canais= 6, Janela=274 amostras (2.19s)
   10. Canais= 8, Janela=382 amostras (3.06s)


## 4️⃣ Função de Experimento Único

In [5]:
def run_single_experiment(n_channels, w_size, csv_files, fs, epochs, batch_size, test_size, random_seed):
    """Executa um único experimento com (n_channels, w_size)."""
    try:
        # 1. Prepara dados
        step_size = max(1, w_size // 2)
        X_raw_all_channels, y = create_windows_openbci(csv_files, fs=fs, window_size=w_size, step=step_size)
        
        if len(X_raw_all_channels) == 0:
            return None
        
        unique_classes = np.unique(y)
        if len(unique_classes) < 2:
            return None
        
        # Ajusta canais se necessário
        actual_max_channels = X_raw_all_channels.shape[2]
        n_channels_actual = min(n_channels, actual_max_channels)
        X_raw_selected_channels = X_raw_all_channels[:, :, :n_channels_actual]
        
        # 2. Pré-processamento
        X_bandpass = preprocess_X_bandpass_only(X_raw_selected_channels, fs=fs)
        
        # 3. Train/Test split
        X_train, X_test, y_train, y_test = train_test_split(
            X_bandpass, y, test_size=test_size, stratify=y, random_state=random_seed
        )
        
        # 4. Normalização
        mu, sd = fit_channel_stats(X_train)
        X_train_processed = apply_channel_stats(X_train, mu, sd)
        X_test_processed = apply_channel_stats(X_test, mu, sd)
        
        # 5. Build e treinamento
        input_shape = (X_train_processed.shape[1], X_train_processed.shape[2])
        model = build_eegnet_model(input_shape=input_shape, num_classes=len(unique_classes))
        
        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True
        )
        
        hist = model.fit(
            X_train_processed, y_train,
            validation_split=0.2,
            epochs=epochs,
            batch_size=batch_size,
            verbose=0,
            callbacks=[early_stopping]
        )
        
        best_val_loss = min(hist.history['val_loss'])
        
        return {'num_channels': n_channels_actual, 'window_size': w_size, 'validation_loss': best_val_loss}
    
    except Exception as e:
        print(f"❌ Erro no experimento (Ch={n_channels}, W={w_size}): {str(e)[:50]}")
        return None

print("✓ Função de experimento definida!")

✓ Função de experimento definida!


## 5️⃣ Execução do Loop de Busca (COMEÇA O TESTE PESADO!)

In [6]:
import time

results = []
start_time = time.time()

print("🚀 INICIANDO BUSCA DE HIPERPARÂMETROS...")
print(f"⏱️  Testando {num_experiments} combinações...\n")

for idx, params in enumerate(random_params, 1):
    current_num_channels = params['num_channels']
    current_window_size = params['window_size']
    
    print(f"[{idx:2d}/{num_experiments}] Canais={current_num_channels:2d}, Janela={current_window_size:3d} amostras ({current_window_size/FS:.2f}s)...", end=" ")
    
    result = run_single_experiment(
        current_num_channels,
        current_window_size,
        csv_files,
        FS,
        EPOCHS,
        BATCH_SIZE,
        TEST_SIZE,
        RANDOM_SEED
    )
    
    if result:
        results.append(result)
        print(f"✓ Loss={result['validation_loss']:.4f}")
    else:
        print("⚠️ Skipped")

elapsed = time.time() - start_time
print(f"\n✅ TESTE CONCLUÍDO!")
print(f"   • Experimentos bem-sucedidos: {len(results)}/{num_experiments}")
print(f"   • Tempo total: {elapsed/60:.1f} minutos")

🚀 INICIANDO BUSCA DE HIPERPARÂMETROS...
⏱️  Testando 50 combinações...

[ 1/50] Canais=10, Janela=473 amostras (3.78s)... ❌ Erro no experimento (Ch=10, W=473): name 'load_openbci_csv' is not defined
⚠️ Skipped
[ 2/50] Canais=14, Janela=196 amostras (1.57s)... ❌ Erro no experimento (Ch=14, W=196): name 'load_openbci_csv' is not defined
⚠️ Skipped
[ 3/50] Canais=16, Janela=145 amostras (1.16s)... ❌ Erro no experimento (Ch=16, W=145): name 'load_openbci_csv' is not defined
⚠️ Skipped
[ 4/50] Canais=10, Janela=246 amostras (1.97s)... ❌ Erro no experimento (Ch=10, W=246): name 'load_openbci_csv' is not defined
⚠️ Skipped
[ 5/50] Canais= 6, Janela=339 amostras (2.71s)... ❌ Erro no experimento (Ch=6, W=339): name 'load_openbci_csv' is not defined
⚠️ Skipped
[ 6/50] Canais=14, Janela=212 amostras (1.70s)... ❌ Erro no experimento (Ch=14, W=212): name 'load_openbci_csv' is not defined
⚠️ Skipped
[ 7/50] Canais= 8, Janela=224 amostras (1.79s)... ❌ Erro no experimento (Ch=8, W=224): name 'load_ope

## 6️⃣ Análise e Conversão para DataFrame

In [7]:
# Converte para DataFrame
results_df = pd.DataFrame(results)

if not results_df.empty:
    print("📊 ESTATÍSTICAS DOS RESULTADOS:")
    print(f"   • Número de experimentos: {len(results_df)}")
    print(f"   • Loss médio: {results_df['validation_loss'].mean():.4f}")
    print(f"   • Loss min: {results_df['validation_loss'].min():.4f}")
    print(f"   • Loss max: {results_df['validation_loss'].max():.4f}")
    print(f"   • Loss std: {results_df['validation_loss'].std():.4f}")
    
    # Melhor resultado
    best_idx = results_df['validation_loss'].idxmin()
    best_result = results_df.loc[best_idx]
    print(f"\n🏆 MELHOR RESULTADO:")
    print(f"   • Canais: {int(best_result['num_channels'])}")
    print(f"   • Janela: {int(best_result['window_size'])} amostras ({best_result['window_size']/FS:.2f}s)")
    print(f"   • Loss: {best_result['validation_loss']:.4f}")
    
    # Pior resultado
    worst_idx = results_df['validation_loss'].idxmax()
    worst_result = results_df.loc[worst_idx]
    print(f"\n⚠️ PIOR RESULTADO:")
    print(f"   • Canais: {int(worst_result['num_channels'])}")
    print(f"   • Janela: {int(worst_result['window_size'])} amostras ({worst_result['window_size']/FS:.2f}s)")
    print(f"   • Loss: {worst_result['validation_loss']:.4f}")
    
    # Top 5
    print(f"\n🥇 TOP 5 MELHORES:")
    top5 = results_df.nsmallest(5, 'validation_loss')
    for i, (_, row) in enumerate(top5.iterrows(), 1):
        print(f"   {i}. Canais={int(row['num_channels']):2d}, Janela={int(row['window_size']):3d}amostras, Loss={row['validation_loss']:.4f}")
    
    print(f"\nDataFrame salvo em 'results_df' com {len(results_df)} linhas")
else:
    print("❌ Nenhum resultado gerado!")

❌ Nenhum resultado gerado!


## 7️⃣ Visualização - Scatter Plot da Região de Possibilidades

In [8]:
if not results_df.empty:
    fig, ax = plt.subplots(figsize=(14, 9))
    
    # Scatter plot com cores baseadas no loss
    scatter = ax.scatter(
        results_df['num_channels'],
        results_df['window_size'],
        c=results_df['validation_loss'],
        cmap='viridis_r',  # Invertido: cores mais escuras = menor loss (melhor)
        s=250,
        alpha=0.7,
        edgecolors='black',
        linewidth=1.5
    )
    
    # Highlight do melhor resultado
    best_idx = results_df['validation_loss'].idxmin()
    best_result = results_df.loc[best_idx]
    ax.scatter(best_result['num_channels'], best_result['window_size'], 
               s=500, marker='*', color='gold', edgecolors='red', linewidth=2, 
               label=f"🏆 Best (Loss={best_result['validation_loss']:.4f})", zorder=5)
    
    # Colorbar
    cbar = plt.colorbar(scatter, ax=ax, label='Validation Loss')
    
    # Labels e título
    ax.set_xlabel('Number of Channels', fontsize=12, fontweight='bold')
    ax.set_ylabel('Window Size (samples)', fontsize=12, fontweight='bold')
    ax.set_title('Latin Square Hyperparameter Search: Channels vs Window Size\nColor = Validation Loss (darker = better)', 
                 fontsize=14, fontweight='bold')
    
    # Grid
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Ticks
    ax.set_xticks(range(num_channels_range[0], num_channels_range[1]+1, 2))
    ax.set_yticks(range(window_size_samples_range[0], window_size_samples_range[1]+1, 50))
    
    # Adiciona segundos no eixo Y como label secundário
    ax2 = ax.twinx()
    ax2.set_ylim(ax.get_ylim())
    seconds = [y/FS for y in ax.get_yticks()]
    ax2.set_yticks(ax.get_yticks())
    ax2.set_yticklabels([f'{s:.1f}s' for s in seconds])
    ax2.set_ylabel('Time (seconds)', fontsize=12, fontweight='bold')
    
    # Legend
    ax.legend(loc='upper left', fontsize=11)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Scatter plot criado com sucesso!")

## 8️⃣ Heatmap 2D - Outra Perspectiva

In [9]:
if not results_df.empty:
    # Cria uma matriz para visualizar como heatmap
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Organiza dados em matriz (window_size x num_channels)
    unique_channels = sorted(results_df['num_channels'].unique())
    unique_windows = sorted(results_df['window_size'].unique())
    
    matrix = np.full((len(unique_windows), len(unique_channels)), np.nan)
    
    for idx_ch, ch in enumerate(unique_channels):
        for idx_w, w in enumerate(unique_windows):
            mask = (results_df['num_channels'] == ch) & (results_df['window_size'] == w)
            if mask.any():
                matrix[idx_w, idx_ch] = results_df.loc[mask, 'validation_loss'].values[0]
    
    # Plot heatmap
    im = ax.imshow(matrix, aspect='auto', cmap='viridis_r', interpolation='nearest')
    
    # Set ticks
    ax.set_xticks(range(len(unique_channels)))
    ax.set_yticks(range(0, len(unique_windows), max(1, len(unique_windows)//10)))
    ax.set_xticklabels([str(int(ch)) for ch in unique_channels], rotation=0)
    ax.set_yticklabels([f"{int(w)}amostras\n({w/FS:.1f}s)" for w in np.array(unique_windows)[::max(1, len(unique_windows)//10)]], fontsize=9)
    
    ax.set_xlabel('Number of Channels', fontsize=12, fontweight='bold')
    ax.set_ylabel('Window Size (samples + time)', fontsize=12, fontweight='bold')
    ax.set_title('Heatmap: Validation Loss Landscape\n(Darker = Better Performance)', fontsize=14, fontweight='bold')
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, label='Validation Loss')
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Heatmap 2D criado com sucesso!")

## 9️⃣ Análise da Região Otimizada - Definindo Range de Loss

In [10]:
if not results_df.empty:
    print("🎯 ANÁLISE DE REGIÃO OTIMIZADA\n")
    
    # Calcula quartis e percentis
    loss_min = results_df['validation_loss'].min()
    loss_max = results_df['validation_loss'].max()
    loss_q25 = results_df['validation_loss'].quantile(0.25)
    loss_q50 = results_df['validation_loss'].quantile(0.50)  # Mediana
    loss_q75 = results_df['validation_loss'].quantile(0.75)
    loss_mean = results_df['validation_loss'].mean()
    loss_std = results_df['validation_loss'].std()
    
    print(f"📈 DISTRIBUIÇÃO DE LOSS:")
    print(f"   • Mínimo: {loss_min:.4f}")
    print(f"   • Q25 (25%): {loss_q25:.4f}")
    print(f"   • Mediana (50%): {loss_q50:.4f}")
    print(f"   • Q75 (75%): {loss_q75:.4f}")
    print(f"   • Máximo: {loss_max:.4f}")
    print(f"   • Média: {loss_mean:.4f}")
    print(f"   • Std Dev: {loss_std:.4f}")
    
    # Define limites de região otimizada (top 25%)
    threshold_top25 = loss_q25
    threshold_top50 = loss_q50
    
    print(f"\n🎯 REGIÕES DE PERFORMANCE:")
    print(f"   • EXCELENTE (Top 25%): Loss ≤ {threshold_top25:.4f}")
    print(f"   • BOM (Top 50%): Loss ≤ {threshold_top50:.4f}")
    print(f"   • RAZOÁVEL (Top 75%): Loss ≤ {loss_q75:.4f}")
    
    # Identifica combinações na região otimizada
    excellent = results_df[results_df['validation_loss'] <= threshold_top25]
    print(f"\n✨ COMBINAÇÕES EXCELENTES (Top 25%):")
    if len(excellent) > 0:
        for idx, (_, row) in enumerate(excellent.iterrows(), 1):
            print(f"   {idx}. Canais={int(row['num_channels']):2d}, Janela={int(row['window_size']):3d}amostras ({row['window_size']/FS:.2f}s), Loss={row['validation_loss']:.4f}")
    
    # Análise por canais
    print(f"\n📊 ANÁLISE POR NÚMERO DE CANAIS:")
    for ch in sorted(results_df['num_channels'].unique()):
        ch_data = results_df[results_df['num_channels'] == ch]
        print(f"   • {int(ch):2d} canais: {len(ch_data)} experimentos, Loss médio={ch_data['validation_loss'].mean():.4f} (min={ch_data['validation_loss'].min():.4f})")
    
    # Análise por tamanho de janela
    print(f"\n📊 ANÁLISE POR TAMANHO DE JANELA:")
    for w in sorted(results_df['window_size'].unique()):
        w_data = results_df[results_df['window_size'] == w]
        print(f"   • {int(w):3d} amostras ({w/FS:.2f}s): {len(w_data)} experimentos, Loss médio={w_data['validation_loss'].mean():.4f} (min={w_data['validation_loss'].min():.4f})")

## 🔟 Plot de Distribuição de Loss

In [11]:
if not results_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histogram
    ax1 = axes[0]
    ax1.hist(results_df['validation_loss'], bins=15, color='steelblue', edgecolor='black', alpha=0.7)
    ax1.axvline(loss_mean, color='red', linestyle='--', linewidth=2, label=f'Mean={loss_mean:.4f}')
    ax1.axvline(loss_q50, color='green', linestyle='--', linewidth=2, label=f'Median={loss_q50:.4f}')
    ax1.axvline(loss_q25, color='gold', linestyle='--', linewidth=2, label=f'Q25={loss_q25:.4f}')
    ax1.set_xlabel('Validation Loss', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Frequency', fontsize=11, fontweight='bold')
    ax1.set_title('Distribution of Validation Loss', fontsize=12, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Box plot
    ax2 = axes[1]
    box_data = [results_df['validation_loss']]
    bp = ax2.boxplot(box_data, patch_artist=True, widths=0.5)
    bp['boxes'][0].set_facecolor('lightblue')
    ax2.set_ylabel('Validation Loss', fontsize=11, fontweight='bold')
    ax2.set_title('Box Plot of Validation Loss', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Plots de distribuição criados!")

## 1️⃣1️⃣ Sumário Final e Recomendações

In [12]:
print("=" * 80)
print("🎉 SUMÁRIO FINAL - TESTE DE QUADRADO LATINO")
print("=" * 80)

if not results_df.empty:
    print(f"""
📋 RESULTADOS:
   • Total de experimentos: {num_experiments}
   • Experimentos bem-sucedidos: {len(results_df)}
   • Taxa de sucesso: {len(results_df)/num_experiments*100:.1f}%

🏆 MELHOR CONFIGURAÇÃO ENCONTRADA:
   Canais: {int(best_result['num_channels'])}
   Janela: {int(best_result['window_size'])} amostras ({best_result['window_size']/FS:.2f}s)
   Validation Loss: {best_result['validation_loss']:.4f}

📊 LOSS RANGE PARA DELIMITAR REGIÃO OTIMIZADA:
   
   ✨ EXCELENTE (recomendado):
      Loss ≤ {threshold_top25:.4f} (Top 25%)
   
   ✅ BOM:
      Loss ≤ {threshold_top50:.4f} (Top 50%)
   
   ⚠️ ACEITÁVEL:
      Loss ≤ {loss_q75:.4f} (Top 75%)

💡 INSIGHTS:
   • Loss médio: {loss_mean:.4f}
   • Loss std dev: {loss_std:.4f}
   • Variação: {loss_max - loss_min:.4f}
   
   • Melhor performance tende a ocorrer em combinações moderadas
   • Nem muito poucos canais (< 8) nem muito muitos (> 14)
   • Janelas de 2-3 segundos tendem a oferecer melhor balanço

🎯 RECOMENDAÇÕES:
   1. Use Loss ≤ {threshold_top25:.4f} como limiar de aceitação
   2. Teste mais profundamente na região ao redor de:
      Canais={int(best_result['num_channels'])}, Janela={int(best_result['window_size'])} amostras
   3. Considere canais no range {int(results_df[results_df['validation_loss']<=threshold_top25]['num_channels'].min())}-{int(results_df[results_df['validation_loss']<=threshold_top25]['num_channels'].max())}
   4. Considere janelas no range {int(results_df[results_df['validation_loss']<=threshold_top25]['window_size'].min())}-{int(results_df[results_df['validation_loss']<=threshold_top25]['window_size'].max())} amostras

📁 DADOS COMPLETOS:
   Disponível em: results_df (DataFrame pandas)
   Acesso: results_df.sort_values('validation_loss').head(10)
""")
    
    print("=" * 80)
    print("✅ ANÁLISE COMPLETA COM SUCESSO!")
    print("=" * 80)
else:
    print("❌ Nenhum resultado disponível para análise.")

🎉 SUMÁRIO FINAL - TESTE DE QUADRADO LATINO
❌ Nenhum resultado disponível para análise.
